<a href="https://colab.research.google.com/github/nohaelkachach/casiav2-splicing-gradcam-audit/blob/main/notebooks/01_data_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Data validation

Validates the CASIA v2 dataset (images, groundtruth masks) against the authoritative file lists (`au_list.txt`, `tp_list.txt`) provided alongside the dataset, following the corrected naming convention documented by Pham et al. (2019).

**Expected final counts:** 7,491 authentic images, 5,123 tampered images (3,295 copy-move + 1,828 spliced), 5,123 groundtruth masks (1,828 spliced + 3,295 copy-move).

**Sources:**
- Dong, Wang & Tan (2013) — CASIA v2 base dataset
- Pham, Lee, Kwon & Park (2019), *Symmetry* — groundtruth masks and corrected file naming

## Mount Drive and set paths
Adjust `base` and `gt_dir` to match your own Drive layout.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
from collections import Counter

base = "/content/drive/MyDrive/CASIA2.0/CASIA2.0_revised"  # adjust to your actual path
au_dir = os.path.join(base, "Au")
tp_dir = os.path.join(base, "Tp")
gt_dir = "/content/drive/MyDrive/CASIA2.0/CASIA2.0_Groundtruth/CASIA2.0_Groundtruth"  # adjust if the real path differs

Mounted at /content/drive


## Step 1 — Raw file counts
Baseline counts before any cleaning. These are expected to differ slightly from the final validated counts due to system files and download-duplicate artifacts (see Step 2).

In [2]:
au_files = sorted(os.listdir(au_dir))
tp_files = sorted(os.listdir(tp_dir))

print(f"Authentic images: {len(au_files)}")
print(f"Tampered images: {len(tp_files)}")

Authentic images: 7492
Tampered images: 5123


## Step 2 — Validate against authoritative file lists
`au_list.txt` and `tp_list.txt` are the dataset's authoritative inventories. Filtering the raw folder contents against these lists removes system files (e.g. `Thumbs.db`) and download-duplicate artifacts.

**Expected: 7,491 authentic, 5,123 tampered.**

In [3]:
with open(os.path.join(base, "au_list.txt")) as f:
    au_list_content = [line.strip() for line in f if line.strip()]
with open(os.path.join(base, "tp_list.txt")) as f:
    tp_list_content = [line.strip() for line in f if line.strip()]

au_files_final = sorted(set(au_list_content) & set(au_files))
tp_files_final = sorted(set(tp_list_content) & set(tp_files))

print(f"Final authentic count: {len(au_files_final)}")   # expect 7491
print(f"Final tampered count: {len(tp_files_final)}")     # expect 5123

Final authentic count: 7491
Final tampered count: 5123


## Step 3 — Category and tampering-type breakdown
`D` = spliced (different source image), `S` = copy-move (same source image), per the dataset's naming convention.

**Expected tampering split: 3,295 copy-move (S), 1,828 spliced (D).**

In [4]:
au_categories = Counter()
for f in au_files_final:
    parts = f.split("_")
    if len(parts) >= 2:
        au_categories[parts[1]] += 1
print("Authentic categories:", au_categories)

tp_type = Counter()
for f in tp_files_final:
    parts = f.split("_")
    tp_type[parts[1]] += 1
print("Tampering type breakdown:", tp_type)   # expect {'S': 3295, 'D': 1828}

Authentic categories: Counter({'ani': 1094, 'arc': 1075, 'nat': 1008, 'cha': 980, 'pla': 976, 'sec': 953, 'art': 913, 'ind': 346, 'txt': 146})
Tampering type breakdown: Counter({'S': 3295, 'D': 1828})


## Step 4 — Audit for junk files and duplicates
Confirms the validated sets contain only expected image extensions and no case-insensitive duplicate filenames.

In [5]:
def audit(files, label):
    ext_counts = Counter(os.path.splitext(f)[1].lower() for f in files)
    dupes = [n for n, c in Counter(f.lower() for f in files).items() if c > 1]
    print(f"--- {label} ---")
    print("Extensions:", ext_counts)
    print(f"Case-insensitive duplicate names: {len(dupes)}")

audit(au_files_final, "Authentic (final)")
audit(tp_files_final, "Tampered (final)")

--- Authentic (final) ---
Extensions: Counter({'.jpg': 7437, '.bmp': 54})
Case-insensitive duplicate names: 0
--- Tampered (final) ---
Extensions: Counter({'.tif': 3059, '.jpg': 2064})
Case-insensitive duplicate names: 0


## Step 5 — Groundtruth mask inventory
Raw mask count before filtering. Expected to include duplicate-download artifacts (see Step 7).

In [6]:
gt_files = sorted(os.listdir(gt_dir))
print(f"Groundtruth mask count: {len(gt_files)}")

Groundtruth mask count: 5123


## Step 6 — Spliced-image mask pairing
Only spliced (`D`) images are used in the downstream analysis. Confirms every spliced image has a matching groundtruth mask.

**Expected: 1,828 spliced files, 0 missing masks.**

In [7]:
spliced_files = [f for f in tp_files_final if f.split("_")[1] == "D"]
print(f"Spliced-only file count: {len(spliced_files)}")  # expect 1828

def expected_mask_name(image_filename):
    stem = os.path.splitext(image_filename)[0]
    return f"{stem}_gt.png"

expected_masks = {f: expected_mask_name(f) for f in spliced_files}
gt_files_set = set(gt_files)

missing_masks = [img for img, mask in expected_masks.items() if mask not in gt_files_set]
print(f"\nSpliced images WITHOUT a matching mask: {len(missing_masks)}")
print(missing_masks[:15])

image_derived_masks = set(expected_masks.values())
orphan_masks = [m for m in gt_files if m not in image_derived_masks]
print(f"\nMasks with no matching spliced image: {len(orphan_masks)}")
print(orphan_masks[:15])

Spliced-only file count: 1828

Spliced images WITHOUT a matching mask: 0
[]

Masks with no matching spliced image: 3295
['Tp_S_CND_L_N_cha00084_cha00084_10203_gt.png', 'Tp_S_CND_M_B_sec00019_sec00019_00040_gt.png', 'Tp_S_CND_M_N_cha00079_cha00079_10195_gt.png', 'Tp_S_CND_M_N_cha00088_cha00088_10174_gt.png', 'Tp_S_CND_M_N_pla00022_pla00022_10969_gt.png', 'Tp_S_CND_M_N_sec00099_sec00099_10365_gt.png', 'Tp_S_CND_S_B_ani00022_ani00022_00142_gt.png', 'Tp_S_CND_S_B_cha00088_cha00088_10179_gt.png', 'Tp_S_CND_S_B_nat00039_nat00039_00958_gt.png', 'Tp_S_CND_S_B_sec00085_sec00085_00105_gt.png', 'Tp_S_CND_S_B_sec00097_sec00097_00117_gt.png', 'Tp_S_CND_S_N_ani00009_ani00009_00129_gt.png', 'Tp_S_CND_S_N_arc00017_arc00017_01115_gt.png', 'Tp_S_CND_S_N_art00044_art00044_10406_gt.png', 'Tp_S_CND_S_N_art00047_art00047_10507_gt.png']


## Step 7 — Copy-move mask pairing
Runs before mask cleanup, since duplicate-mask removal (Step 8) depends on `cm_mask_names` defined here.

**Expected: 3,295 copy-move images, 3,295 matching masks.** The count of "truly orphaned" masks is not a fixed dataset property — it depends on whether this specific download picked up duplicate-download artifacts (e.g. files ending in ` (1).png`). A clean download will show 0 orphaned masks; a download with leftover duplicates may show a handful. Either outcome is valid — check Step 8's deletion count to see which applies to your copy.

In [8]:
copymove_files = [f for f in tp_files_final if f.split("_")[1] == "S"]
print(f"Copy-move image count: {len(copymove_files)}")  # expect 3295

expected_cm_masks = {f: expected_mask_name(f) for f in copymove_files}
cm_mask_names = set(expected_cm_masks.values())

matched_cm_masks = [m for m in gt_files if m in cm_mask_names]
print(f"Copy-move masks matching a real image: {len(matched_cm_masks)}")

all_expected = set(expected_masks.values()) | cm_mask_names
truly_orphaned = [m for m in gt_files if m not in all_expected]
print(f"Truly orphaned masks (no image at all): {len(truly_orphaned)}")
print(truly_orphaned[:15])

Copy-move image count: 3295
Copy-move masks matching a real image: 3295
Truly orphaned masks (no image at all): 0
[]


## Step 8 — Remove duplicate mask files
Deletes any orphaned duplicate-download masks identified in Step 7, leaving only the 5,123 authoritative masks. A deletion count of 0 is a valid, clean outcome — it means this download did not pick up duplicate-download artifacts.

**Run this cell only after confirming Step 7's orphan list (if non-empty) looks like duplicate-download artifacts (e.g. filenames ending in ` (1)`), not genuinely unexplained files.**

In [9]:
duplicate_masks = [f for f in gt_files if f not in (set(expected_masks.values()) | cm_mask_names)]
print(f"Deleting {len(duplicate_masks)} duplicate mask files...")

for f in duplicate_masks:
    os.remove(os.path.join(gt_dir, f))

print("Done. Groundtruth folder now contains only the 5,123 authoritative masks.")

Deleting 0 duplicate mask files...
Done. Groundtruth folder now contains only the 5,123 authoritative masks.
